In [13]:
# Define the custom class
class OneVsRestClassifierParallel:
    def __init__(self, estimator, n_jobs=-1):
        self.estimator = estimator
        self.n_jobs = n_jobs
        self.classifiers_ = []

    def fit(self, X, y):
        import numpy as np
        from sklearn.base import clone
        from tqdm import tqdm
        n_classes = y.shape[1]
        self.classifiers_ = [None] * n_classes
        for i in tqdm(range(n_classes), desc="Training Progress"):
            clf = clone(self.estimator)
            clf.fit(X, y[:, i])
            self.classifiers_[i] = clf
        return self

    def predict(self, X):
        import numpy as np
        predictions = np.array([clf.predict(X) for clf in self.classifiers_]).T
        return predictions

    def predict_proba(self, X):
        import numpy as np
        prob_predictions = np.array([clf.predict_proba(X)[:, 1] for clf in self.classifiers_]).T
        return prob_predictions

# Now load your model
import pickle

with open('model2.pkl', 'rb') as file:
    clf = pickle.load(file)

with open('embedding.pkl', 'rb') as file:
    model = pickle.load(file)

with open('one_hot.pkl', 'rb') as file:
    onehot_encoder = pickle.load(file)


In [25]:
import numpy as np

def predict_top_3(model, input_string, onehot_encoder, text_to_embedding_func):
    """
    Predicts the top 3 most likely classes for a given input string.

    Parameters:
    - model: Trained OneVsRestClassifierParallel model
    - input_string: The text input to classify
    - onehot_encoder: OneHotEncoder used for training (to decode predictions)
    - text_to_embedding_func: Function to convert text to embeddings (ensure it matches training embeddings)

    Returns:
    - List of tuples (class_label, probability)
    """
    # Convert input string to embeddings
    X_input = text_to_embedding_func(input_string).reshape(1, -1)  # Ensure correct shape

    # Get probability predictions
    y_proba = model.predict_proba(X_input)

    # Get top 3 class indices
    top_3_indices = np.argsort(y_proba[0])[-3:][::-1]  # Get 3 highest probabilities

    # Convert class indices back to labels
    class_labels = onehot_encoder.categories_[0]  # Get original class labels
    top_3_labels = class_labels[top_3_indices]

    # Get top 3 probabilities
    top_3_probs = y_proba[0][top_3_indices]

    # Return as a list of tuples (label, probability)
    return list(zip(top_3_labels, top_3_probs))

import pandas as pd
TWONTO_classes = pd.read_excel('TWONTO_classes.xlsx')
Label = dict(zip(TWONTO_classes['Classes'], TWONTO_classes['Label']))

# Example usage
input_text = "backflow preventor valve, city water"
top_3_predictions = predict_top_3(clf, input_text, onehot_encoder, model.encode)

# Print results
for rank, (label, prob) in enumerate(top_3_predictions, start=1):
    print(f"{rank}. {Label[label]}: {prob:.4f}")


1. pipe segment: 1.0000
2. backflow preventer: 1.0000
3. valve: 0.8287


In [26]:
Label['http://www.toronto.ca/TWONTO#00696']

KeyError: 'http://www.toronto.ca/TWONTO#00696'